# Part 8 · Build an agent, ship it, then fix its tool layer

**Six steps, about twelve minutes.** A developer builds an agent against GitHub,
AgentRegistry ships it to kagent, and then one field on the gateway takes the same job
from nineteen model round trips down to two. It ends with two agents sharing one GitHub
integration and having different permissions, enforced on their own identities.

The agent is **Java on Google ADK**, because the tool layer is not the agent's problem
and not the agent language's problem either, and that is easier to believe when the
agent is not Python.

| | Step | Roughly |
|---|---|---|
| 0 | Seed the frozen pull requests (once, not on stage) | 2 min |
| 1 | What one MCP server costs you | 1 min |
| 2 | The gateway holds the credential | 1 min |
| 3 | The catalogue, the skill, and the agent | 3 min |
| 4 | Publish it, deploy it, run it | 3 min |
| 5 | One field: nineteen round trips become two | 2 min |
| 6 | Two agents, one integration, different permissions | 3 min |

Runs on **`mesh1`** and needs the Part 4 platform
(`demo-scripts/agentregistry/setup-mesh1.sh`) plus a GitHub PAT.

**The data is frozen on purpose.** Reading live pull requests from a busy upstream repo
is a bad bet on a projector: the answer changes hour to hour, the numbers stop matching
your slides, and the model sometimes over-fetches and trips code mode's call cap. Step 0
seeds a repo you own with exactly twenty four open pull requests in known states, so the
report is the same every time and "all open pull requests" is always twenty four.

The two claims that hold on every single run, and the only two to build on:

1. Fourteen to nineteen tool calls become two.
2. Fifty to eighty thousand bytes through the model become a couple of dozen.

Read both off the trace on the day. They are an order of magnitude apart every run,
which is the claim; the exact figure is not.

Do not claim it is faster. Both land around thirty seconds and somebody will time you.

## 0. Seed the frozen pull requests

Run this **once**, well before you present. It creates twenty four open pull requests in
a repo you own: two signed off, three drafts, four held, and fifteen waiting on a
sign-off.

The gate is built from three things a personal access token can genuinely produce, and
all three are ordinary in real repositories:

| verdict | how it is produced |
|---|---|
| `draft` | the pull request is opened as a draft |
| `on hold` | the `do-not-merge/hold` label, the Prow convention half the Kubernetes ecosystem uses |
| `no sign-off` | no comment **starting** with `LGTM` |
| `READY` | none of the above |

Be straight about what that gate is when you present it. It is draft status, a hold
label and a sign-off comment. It does **not** look at review approvals, at CI, or at
branch protection, so "ready" here means "passes the demo release gate", not "GitHub
would let you merge it". Seeded pull requests are the point: the same inputs every time,
so the two tool modes can be compared on identical work.

One pull request carries a comment that says "lgtm" in the middle of a sentence while
explicitly declining to sign off, because the rule is `LGTM` at the **start** of a
comment. It is there to catch a report that skim-read the rule, and it has held on every
run.

Two things it deliberately does **not** use. GitHub will not let you approve your own
pull request, and a token cannot write check runs or commit statuses at all, both 403.
`mergeable` was tried and dropped: GitHub computes it lazily, so it returns `null` often
enough to make a live demo unreliable. A label always reads the same.

In [ ]:
# once, not on stage. RESEED=1 closes the demo PRs and starts over.
set -a; . "${SECRETS_FILE:-$HOME/code/solo/secrets/secrets-envs.sh}" 2>/dev/null; set +a
REPO="${DEMO_REPO:-tjorourke/kagent}" ./agents/prtriage/scripts/seed-demo-repo.sh

## Setup and Connect

`setup.sh` is the Part 8 standup and it is **cluster setup, not a demo beat**. It creates
the PAT Secret, the two agentgateway backends (one at the ingress for steps 1 and 2, one
at a waypoint in the mesh for the agents), the waypoint itself, the GAMMA route, and the
catalogue entries. Run it once, well before you present, and never on stage.

The waypoint matters for step 6 and is worth knowing about even though you never touch
it live: it is the only enforcement point that can see a calling agent's own identity.

In [ ]:
# from the folder that holds this notebook (the one containing agents/)
set -a; . "${SECRETS_FILE:-$HOME/code/solo/secrets/secrets-envs.sh}" 2>/dev/null; set +a
source demo-scripts/agentregistry/scripts/connect.sh >/dev/null 2>&1
./agents/prtriage/scripts/setup.sh

In [ ]:
LAB=$PWD
until [ -d "$LAB/agents/prtriage" ]; do
  [ "$LAB" = / ] && { echo "✗ run this from the demo suite folder (the one with agents/)"; break; }
  LAB=$(dirname "$LAB")
done
cd "$LAB"
set -a; . "${SECRETS_FILE:-$HOME/code/solo/secrets/secrets-envs.sh}" 2>/dev/null; set +a
source demo-scripts/agentregistry/scripts/connect.sh
export PART8=agents/prtriage
export LB=$(kubectl --context kind-mesh1 -n agentgateway-system get gateway ar-ingress -o jsonpath='{.status.addresses[0].value}')
export MCP="http://github-mcp.${LB}.sslip.io/"
export KC="kubectl --context kind-mesh1"
export ASK="demo-scripts/agentregistry/scripts/ask.sh"
# The demo reads FROZEN pull requests from a repo you own, so the report is identical
# every run. See agents/prtriage/scripts/seed-demo-repo.sh for how they are built.
export DEMO_REPO="${DEMO_REPO:-tjorourke/kagent}"
# The question, defined here rather than half way down the notebook, so any later cell
# can be re-run on its own. Defined mid-notebook it is empty after a kernel restart,
# and the agent then either fails with "text content blocks must be non-empty" or asks
# which repository you meant, neither of which helps while you are standing up.
export Q="Give me the release report for ${DEMO_REPO}, all open pull requests."
echo
printf "  %-20s %s\n" "kagent UI:"        "http://${KAGENT_UI_HOST}  (admin-user / password)"
printf "  %-20s %s\n" "AgentRegistry UI:" "http://${AR_HOST}  (admin-user / password)"
printf "  %-20s %s\n" "MCP endpoint:"     "$MCP"
printf "  %-20s %s\n" "demo repo:"        "https://github.com/${DEMO_REPO}/pulls"

### Preflight

One command, ten minutes before you present. It checks the fixture counts, the
catalogue, both agents, both MCP paths, model access, stray policies from a previous
run and whether traces are actually landing. Exit code is non-zero if anything is not
ready, and every failure line says what to do about it.

In [ ]:
$PART8/scripts/preflight.sh

### A tiny MCP client

`mcp.sh` is a few lines of curl: initialize, keep the session id, send one JSON-RPC call.
Used in steps 1 and 6 so you can see the wire rather than a framework's idea of it.

In [ ]:
cat > /tmp/mcp.sh <<'SH'
#!/usr/bin/env bash
# mcp.sh <endpoint> <method> [params-json] — one MCP call, session handled.
#
# The endpoint is a *.<ip>.sslip.io name, which means the address is already in the
# name. We pull it out and hand it to curl with --resolve rather than asking a resolver,
# because plenty of home-router and ISP resolvers refuse to return a private address for
# a public name and the failure looks like the gateway being down.
set -euo pipefail
EP="$1"; METHOD="$2"; PARAMS="${3:-}"
HOST=$(printf '%s' "$EP" | sed -E 's#^https?://##; s#[:/].*$##')
IP=$(printf '%s' "$HOST" | sed -nE 's#.*[.]([0-9]{1,3}[.][0-9]{1,3}[.][0-9]{1,3}[.][0-9]{1,3})[.]sslip[.]io$#\1#p')
PORT=$(printf '%s' "$EP" | sed -nE 's#^https?://[^:/]+:([0-9]+).*#\1#p'); PORT=${PORT:-80}
RESOLVE=(); [ -n "$IP" ] && RESOLVE=(--resolve "$HOST:$PORT:$IP")
H=(-H "Content-Type: application/json" -H "Accept: application/json, text/event-stream")
HDR=$(mktemp)
curl -s -m 60 -X POST "$EP" "${RESOLVE[@]}" "${H[@]}" -D "$HDR" -o /dev/null \
  -d '{"jsonrpc":"2.0","id":1,"method":"initialize","params":{"protocolVersion":"2025-06-18","capabilities":{},"clientInfo":{"name":"demo8","version":"1"}}}'
SID=$(grep -i '^mcp-session-id:' "$HDR" | tr -d '\r' | awk '{print $2}')
BODY=$(python3 -c 'import json,sys;print(json.dumps({"jsonrpc":"2.0","id":2,"method":sys.argv[1],"params":json.loads(sys.argv[2] or "{}")}))' "$METHOD" "$PARAMS")
curl -s -m 180 -X POST "$EP" "${RESOLVE[@]}" "${H[@]}" ${SID:+-H "Mcp-Session-Id: $SID"} -d "$BODY" \
  | sed 's/^data: //' | grep -v '^event:' | grep -v '^$'
SH
chmod +x /tmp/mcp.sh; echo "wrote /tmp/mcp.sh"

## 1. What one MCP server costs you

GitHub's hosted MCP server. One server. This is what it puts in the model's context on
every single turn, and what it lets the agent do.

In [ ]:
/tmp/mcp.sh "$MCP" tools/list > /tmp/tools-standard.json
python3 - /tmp/tools-standard.json <<'PY'
import json,sys
d=json.load(open(sys.argv[1])); t=d["result"]["tools"]
WRITE=("create_","update_","delete_","merge_","push_","add_","fork_","_write","request_copilot_review")
w=sorted(x["name"] for x in t if any(k in x["name"] for k in WRITE))
print("tools:                %d" % len(t))
print("tools/list payload:   %d bytes" % len(json.dumps(d)))
print("of which can write:   %d" % len(w))
print()
print("the write tools your agent just acquired:")
print("  " + ", ".join(w))
PY

And the token cost, from Anthropic's own counter rather than an estimate:

In [ ]:
python3 - /tmp/tools-standard.json <<'PY' > /tmp/count.json
import json,sys
d=json.load(open(sys.argv[1]))
tools=[{"name":t["name"],"description":t.get("description",""),
        "input_schema":t.get("inputSchema",{"type":"object"})} for t in d["result"]["tools"]]
print(json.dumps({"model":"claude-sonnet-4-5","tools":tools,"messages":[{"role":"user","content":"hi"}]}))
PY
curl -s https://api.anthropic.com/v1/messages/count_tokens \
  -H "x-api-key: $ANTHROPIC_API_KEY" -H "anthropic-version: 2023-06-01" \
  -H "content-type: application/json" -d @/tmp/count.json

**14,572 tokens before anyone types anything, and 17 of the 44 tools can write**:
`delete_file`, `push_files`, `merge_pull_request`. A token scope cannot say "this one
agent may only read pull requests", because a scope is coarse and the token is shared by
everyone who uses it.

You can of course filter the tool list inside each agent, and a careful team will. The
difference here is that the platform does it once, outside the agent, consistently for
every agent, and it also refuses a direct call that skips the model's tool list
altogether. That last part is the bit agent-side filtering cannot do.

## 2. The gateway holds the credential

`setup.sh` applied this already. Two fields carry the idea: `protocol: StreamableHTTP`
reaches GitHub's hosted server, and `policies.auth.secretRef` injects the PAT upstream
from a Secret the gateway reads.

In [ ]:
# just the backend document (the HTTPRoute after it is ordinary Gateway API)
sed -n '/^apiVersion/,$p' $PART8/yaml/10-github-backend.yaml | sed '/^---$/,$d'

Proof rather than assertion. This call sends **no `Authorization` header**:

In [ ]:
echo "  == the client sends Content-Type and Accept, and nothing else =="
OWNER="${DEMO_REPO%/*}"; NAME="${DEMO_REPO#*/}"
/tmp/mcp.sh "$MCP" tools/call \
  "{\"name\":\"list_pull_requests\",\"arguments\":{\"owner\":\"$OWNER\",\"repo\":\"$NAME\",\"state\":\"open\",\"perPage\":3,\"fields\":[\"number\",\"title\"]}}" \
  | python3 -c "
import json,sys
for r in json.loads(json.load(sys.stdin)['result']['content'][0]['text']):
    print('  #%s  %s' % (r['number'], r['title'][:62]))
"

The agent it is about to serve holds no GitHub credential at all, so it cannot leak one.
That is the first reason to put a gateway here.

## 3. The catalogue, the skill, and the agent

Two things are in the catalogue, and both matter.

The **approved MCP server** points at the gateway, not at `api.githubcopilot.com`, and
specifically at the in-cluster waypoint rather than the ingress hostname. Two reasons,
and step 6 depends on the first: a waypoint is the only place the calling agent's own
identity reaches policy, and it keeps public DNS out of the agent's hot path entirely.

A developer who picks GitHub out of the catalogue therefore gets GitHub through the
enforcement point, and there is no catalogue entry meaning "GitHub, but skip the
gateway".

The **approved skill** is the platform team's write-up of how to drive this gateway: the
parameter is `pullNumber` not `pull_number`, the sandbox has no `Date`, labels come back
as strings or objects, never guess a repository, and do not fetch what cannot change the
answer. Every line is there because a run failed without it. The team learns this once.

In [ ]:
echo "  == the approved MCP server: note the URL =="
arctl get mcpserver github-mcp -o yaml | sed -n '/^spec:/,$p' | grep -E "url:|title:"
echo
echo "  == the approved skill =="
arctl get skill release-report -o yaml | sed -n '/^spec:/,$p' | grep -E "title:|description:" | head -3
echo
echo "  == two of its rules =="
sed -n '/## Do not fetch what cannot change the answer/,+6p' $PART8/skill/release-report/SKILL.md

### The agent is Java, and it is deliberately dull

The whole integration is one method: a `StreamableHttpServerParameters` pointing at the
gateway, an `McpToolset` built from it, and an `LlmAgent` with that toolset. Google ADK,
in Java.

Note what is **not** in it. No GitHub token, no tool list, no policy. The gateway owns
all three, so none of them appear in the agent's code and none of them need a rebuild
when they change.

The only local tool is `today`, and it is there because the gateway's code sandbox
deliberately has no clock, so a program running inside it cannot work out the date.

In [ ]:
# `make show` in java-agent/ prints just this method
sed -n '/private static LlmAgent agent(/,/^  }/p' \
  $PART8/java-agent/src/main/java/io/solo/demo/ReleaseReport.java

`arctl init` cannot scaffold this. As of `v2026.6.1` it does ADK with Python only, and
`--language java` is rejected outright. The catalogue does not care, because an Agent
record just references an image, which is what the next step relies on.

Build it in a container, so nothing on this machine needs a JDK or Maven.

In [ ]:
# The Makefile wraps these two, but they are worth seeing. Maven and the JDK run inside
# the image, which is why nothing on this machine needs Java installed.
# `make skill.md` PULLS the approved skill from AgentRegistry, the same way demo 4
# pulls its skill, so the image is built against the pinned registry record rather
# than whatever is in the working tree. LOCAL_SKILL=1 overrides while editing.
( cd $PART8/java-agent \
    && make skill.md \
    && docker build -t localhost:5001/prtriage-java:latest . \
    && docker push -q localhost:5001/prtriage-java:latest )

## 4. Publish it, deploy it, run it

`make publish` registers the agent in AgentRegistry. `make deploy` applies one
Deployment record naming the agent and the runtime, and AgentRegistry does the rest: it
creates the kagent `Agent`, derives the MCP wiring from the approved server in the
catalogue, and the controller brings the pod up. No Helm, no hand-written pod spec.

`SERVE=true` in that record is what starts the agent's A2A server, so kagent's readiness
probe on `/.well-known/agent-card.json` passes and the controller can reach it. It is the
same shape as any other kagent agent.

Two `arctl` commands. The first is the catalogue entry, the second is the deploy. This is
the same pair a Python agent uses, because the registry cares what an agent is allowed
to call, not what language it is written in.

In [ ]:
echo "  == 1) publish the agent to the catalogue =="
cat $PART8/java-agent/agent.yaml
arctl apply -f $PART8/java-agent/agent.yaml
echo
arctl get agents

Now the deploy. One record naming the agent and the runtime. `SERVE=true` is what starts
the agent's A2A server so kagent's readiness probe on `/.well-known/agent-card.json`
passes.

In [ ]:
echo "  == 2) deploy it onto the kagent runtime =="
cat $PART8/yaml/60-java-deploy-kagent.yaml
arctl apply -f $PART8/yaml/60-java-deploy-kagent.yaml

In [ ]:
# AgentRegistry creates the kagent Agent asynchronously, so wait for the object before
# waiting on its rollout, then for kagent to actually call it Ready: the Deployment being
# up is not the same as the controller having fetched the agent card.
until $KC -n kagent get deploy/prtriagejava >/dev/null 2>&1; do sleep 2; done
$KC -n kagent rollout status deploy/prtriagejava --timeout=240s
$KC -n kagent wait --for=condition=Ready agent/prtriagejava --timeout=120s
echo
$KC -n kagent get agent prtriagejava
$KC -n kagent get pods | grep -E 'NAME|prtriagejava'

### Run it

**On stage, do this in the kagent UI**: open the URL the Connect cell printed, pick
**prtriagejava**, and paste the question. You get the answer and the tool-call span tree
in the Tracing tab, which is the thing worth projecting.

`ask.sh` is the same call headless, through the same OIDC-protected A2A endpoint on the
controller, and it prints the tool-call trace itself. (`make ask` in `java-agent/` is a
shorthand for exactly this.) Note the mode we are in first.

In [ ]:
# Both backends: the ingress one is what /tmp/mcp.sh talks to from the laptop, the
# kagent one is what the agents talk to through the waypoint. They must agree or the
# demo shows one thing and measures another.
for ns in agentgateway-system kagent; do
  $KC -n $ns patch enterpriseagentgatewaybackend github-mcp \
    --type=merge -p '{"spec":{"entMcp":{"toolMode":"Standard"}}}'
done
# The agent lists its MCP tools ONCE at startup, so it has to restart to see a mode
# change. The image does not change; this implementation just caches the tool list.
$PART8/scripts/reload-agent.sh

In [ ]:
AGENT_PREFIX=prtriagejava $ASK "$Q" > /tmp/run-standard.txt 2>&1
tail -32 /tmp/run-standard.txt
echo
echo "  == what that cost =="
$PART8/scripts/trace-cost.sh /tmp/run-standard.txt

Read the trace, not just the answer, and **read the numbers off the screen rather than
from these notes**: the model chooses how to batch, so this lands somewhere between
fourteen and nineteen round trips and fifty to eighty thousand bytes. The shape is what
matters. One call to list the pull requests, then one per pull request to read its
discussion, each re-sending the whole conversation so far, with every raw API response
landing in the context window on the way.

Fewer than twenty five, because the approved skill tells it not to fetch what cannot
change the answer: a draft or a held pull request is already decided, so reading its
comments buys nothing. The registry saves those calls before the gateway does anything.

Now check the report rather than trusting it. `check-report.sh` reads the fixture state
straight from GitHub, by the same three rules the skill gives the agent, and compares it
line by line.

In [ ]:
$PART8/scripts/check-report.sh /tmp/run-standard.txt || true

Do not script what that says. On this cluster the default mode usually gets every
verdict right and miscounts the total, which is a small, harmless and very telling
error: it read twenty four things one at a time and lost track of how many. Sometimes it
gets the count right too.

Either way, the comparison is not about catching the model out. It is about the tool
overhead and where the data is handled, and both reports get checked against the same
fixture.

### The accuracy problem, which is the real reason to care

The frozen data here is small enough that the model mostly copes, and the miscount is
the visible edge of it. Against a live repo it does not cope, and this is measured rather
than argued: running the same question against a busy upstream repository, where each
pull request drags in full check-run JSON and reviewer bodies, the default mode needed
twenty round trips and 87,630 bytes, and **twice in a row it reported pull requests as
ready to merge that had no approval at all**, four in one run and five in the next,
verified against GitHub. Another run silently dropped a pull request and reported seven
of eight.

Nothing was misconfigured and the model was not being stupid. It made twenty separate
calls, and by the time it wrote the report the review data was tens of thousands of
tokens behind it. It lost track.

That is the argument. The default mode is not merely slow, it is **less accurate**, and
it fails in the worst direction: telling you to ship things nobody approved. Do not build
a live demo on the model making that mistake, because it is stochastic. Show it as
evidence and demonstrate the fix.

## 5. One field: nineteen round trips become two

`toolMode: CodeSearch` stops the gateway handing the model 44 tools. It gives it two
instead: `get_tool` to look up an operation's schema, and `run_code` to execute a
JavaScript program against them. The model writes one program, the gateway runs it in a
sandbox, makes the upstream calls, and returns only what the program returns.

Same agent. Same image. Same catalogue. One field on the backend.

(`toolMode: Code` is the conservative sibling: one `run_code` tool with all 44 signatures
baked into its description, so nothing has to be looked up. It costs 6,302 schema tokens
against CodeSearch's 1,300 and lands on the same two round trips. The appendix has both.)

In [ ]:
for ns in agentgateway-system kagent; do
  $KC -n $ns patch enterpriseagentgatewaybackend github-mcp \
    --type=merge -p '{"spec":{"entMcp":{"toolMode":"CodeSearch"}}}'
done
# Ask the gateway what it is serving rather than guessing with a sleep. On a busy
# cluster a guess is too short, the next cell measures the OLD surface, and it reads
# as the gateway ignoring the change.
$PART8/scripts/wait-for-mode.sh CodeSearch
/tmp/mcp.sh "$MCP" tools/list | python3 -c "
import json,sys
t=json.load(sys.stdin)['result']['tools']
print('tools the model now gets:',len(t),'->',', '.join(x['name'] for x in t))
"
$PART8/scripts/reload-agent.sh

In [ ]:
AGENT_PREFIX=prtriagejava $ASK "$Q" > /tmp/run-code.txt 2>&1
tail -32 /tmp/run-code.txt
echo
echo "  == what that cost =="
$PART8/scripts/trace-cost.sh /tmp/run-code.txt

**Two round trips.** `today()`, then one program that made the GitHub calls inside the
gateway and returned the finished report. **Twenty four bytes** crossed the context
window instead of tens of thousands.

Check this one too, the same way.

In [ ]:
$PART8/scripts/check-report.sh /tmp/run-code.txt || true

Same verdicts, and the count is right, because the program counted with `length` rather
than the model keeping a tally across nineteen turns.

Worth saying plainly: code mode does not make the model correct. It writes JavaScript
that can be wrong like anything else. What it does is move the counting and filtering
somewhere explicit and inspectable, and stop the raw data passing through a context
window on the way.

That is the whole idea in one line. Anything the model has to hold is something it can
lose, so the sandbox holds it instead.

Measured on this cluster, twenty four pull requests, two runs each and identical both
times:

| | Standard | CodeSearch |
|---|---|---|
| tools the model holds | 45 | 2 |
| schema tokens per turn | 14,572 | **1,300** |
| model round trips | 14 to 19 | **2** |
| payload through the model | 55,852 to 80,379 B | **24 B** |
| matches the fixture | verdicts yes, count usually wrong | **yes, checked** |

**Do not claim it is faster.** Both land around thirty seconds, because code mode spends
its saving on the model writing the program, and somebody in the room will time you. The
claims that hold every run are the round trips, the bytes and the count.

**What to actually show on the projector:** scroll the Standard trace, which is a
screenful of raw GitHub JSON, then show the CodeSearch trace, which is two lines. That
contrast reads from the back of the room in a way a stopwatch never would, and the byte
count under each makes it a number rather than an impression.

### The sandbox is small on purpose

Worth probing rather than assuming, because it decides what the model is allowed to
write. The only way out of this sandbox is the generated tool functions: there is no
`fetch`, no `require`, no `process`. A program cannot phone home, it can only call
approved tools.

In [ ]:
/tmp/mcp.sh "$MCP" tools/call '{"name":"run_code","arguments":{"code":"const p={}; for (const n of [\"Date\",\"fetch\",\"Math\",\"JSON\",\"Promise\",\"Map\",\"console\",\"process\",\"require\"]) { try { p[n]=typeof eval(n); } catch(e) { p[n]=\"MISSING\"; } } p"}}' \
  | python3 -c "
import json,sys
print(json.dumps(json.loads(json.load(sys.stdin)['result']['content'][0]['text'])['success'],indent=1))
"

No `Date` is why the agent keeps a local `today` tool. No `Map` is the kind of thing a
model reaches for by habit. Both are written down in the approved skill, which is the
only reason the run above worked first time.

There is one more limit worth knowing: **a program may make at most 20 upstream tool
calls.** Twenty four pull requests fit because drafts and held pull requests need no
comment read, which is exactly what the skill tells the agent.

### One more, to answer the heckle

Somebody will assume the program was written in advance. It was not, and the cheapest way
to prove it is to ask something you obviously did not plan for. It writes a different
program and still answers in a turn or two.

Name the repository in the question. Leave it out and the agent has nothing to work from,
which is why the approved skill tells it to ask rather than pick one.

In [ ]:
AGENT_PREFIX=prtriagejava $ASK \
  "On $DEMO_REPO, of the open pull requests that are on hold, which was opened earliest? Answer in one line." \
  | tail -6

## 6. Two agents, one integration, different permissions

The report needs to read pull requests. A release agent needs to merge them. Both use
the **same** approved GitHub integration, the same image and the same skill.

This is enforced at the waypoint, because that is the only place the agent's own
identity is available: a waypoint receives HBONE from ztunnel and can read the peer
certificate, so `source.identity` is who the workload provably is rather than what it
claims. An ingress gateway cannot do this, since by then the connection has left the
mesh. It is the same reason Part 4 enforces its `AccessPolicy` at a waypoint.

One policy, one expression. Two `Allow` policies on one backend would raise a question
about how they combine, and a demo should not depend on the answer.

In [ ]:
cat $PART8/yaml/70-identity-policy.yaml

In [ ]:
$KC apply -f $PART8/yaml/70-identity-policy.yaml
echo
echo "  == and a second agent, same image, different identity =="
arctl apply -f $PART8/yaml/80-release-agent.yaml
until $KC -n kagent get deploy/releasejava >/dev/null 2>&1; do sleep 2; done
$KC -n kagent rollout status deploy/releasejava --timeout=240s
$KC -n kagent wait --for=condition=Ready agent/releasejava --timeout=180s
echo
$KC -n kagent get agent prtriagejava releasejava
echo
echo "  == the identities the policy matches on =="
for a in prtriagejava releasejava; do
  printf "  %-14s serviceAccount=" "$a"
  $KC -n kagent get deploy $a -o jsonpath='{.spec.template.spec.serviceAccountName}{"\n"}'
done

Now ask the same question of the gateway from each agent's own pod. Same URL, same
request body, no credentials anywhere. The only difference is who is asking.

In [ ]:
# The matrix is about which tool NAMES each identity can see, so put the surface back
# to Standard first. In CodeSearch mode the only tools are get_tool and run_code, and
# both agents would look identical whatever the policy says.
for ns in agentgateway-system kagent; do
  $KC -n $ns patch enterpriseagentgatewaybackend github-mcp \
    --type=merge -p '{"spec":{"entMcp":{"toolMode":"Standard"}}}' >/dev/null
done
$PART8/scripts/wait-for-mode.sh Standard
$PART8/scripts/reload-agent.sh >/dev/null
DEP=releasejava $PART8/scripts/reload-agent.sh >/dev/null
$PART8/scripts/identity-matrix.sh

The triage agent cannot see `merge_pull_request` at all. The release agent can. A third
workload in the same namespace, with an identity the policy does not name, gets nothing.

### The enforcement proof

An agent saying "I cannot merge" is the model being agreeable. This sends the request
the model would have made, straight at the gateway from inside the agent's own pod, so
it carries the real identity and nothing else. It goes nowhere near the model.

It targets a pull request number that does not exist on purpose: if policy ever failed
to propagate, the worst case is a 404 rather than a merged fixture, and the error text
tells you which of the two happened.

In [ ]:
echo "### as the TRIAGE agent"
$PART8/scripts/try-merge.sh prtriagejava || true
echo
echo "### as the RELEASE agent"
$PART8/scripts/try-merge.sh releasejava || true

### The same thing in code mode, where it is stronger

In `CodeSearch` the generated API is built **after** the policy is applied, so a denied
operation is not a function in the sandbox at all. The program cannot express the call,
rather than making it and being refused.

In [ ]:
for ns in agentgateway-system kagent; do
  $KC -n $ns patch enterpriseagentgatewaybackend github-mcp \
    --type=merge -p '{"spec":{"entMcp":{"toolMode":"CodeSearch"}}}' >/dev/null
done
$PART8/scripts/wait-for-mode.sh CodeSearch
$PART8/scripts/reload-agent.sh >/dev/null
echo "  == a program that tries to merge, as the TRIAGE identity =="
$KC -n kagent exec deploy/prtriagejava -- sh -c '
  INIT='"'"'{"jsonrpc":"2.0","id":1,"method":"initialize","params":{"protocolVersion":"2025-06-18","capabilities":{},"clientInfo":{"name":"p","version":"1"}}}'"'"'
  U=http://github-mcp.kagent.svc.cluster.local/
  SID=$(wget -qS -O /dev/null --header="Content-Type: application/json" --header="Accept: application/json, text/event-stream" --post-data="$INIT" $U 2>&1 | grep -i mcp-session-id | awk "{print \$2}")
  wget -q --content-on-error -O- --header="Content-Type: application/json" --header="Accept: application/json, text/event-stream" \
    ${SID:+--header="Mcp-Session-Id: $SID"} \
    --post-data='"'"'{"jsonrpc":"2.0","id":9,"method":"tools/call","params":{"name":"run_code","arguments":{"code":"await merge_pull_request({ owner: \"tjorourke\", repo: \"kagent\", pullNumber: 99999 })"}}}'"'"' $U 2>&1 \
    | sed "s/^data: //" | grep . | head -2' 2>&1 | sed 's/^/    /'

Read the two `try-merge` results carefully, because the contrast is the whole point.

The triage agent got `Unknown tool: merge_pull_request`. Not a `403`, and not a refusal
from GitHub: the tool does not exist for that identity, so the request never left the
cluster.

The release agent's error names `api.github.com`. Its request went all the way to
GitHub, and the only thing that stopped it was the pull request not existing.

Same request. Same gateway. Different identity.

And the credential has not moved: the PAT in that Secret can merge and delete files for
either of them. GitHub's permissions bound what the credential can ever do; gateway
policy gives each agent using that one integration a different set of tool permissions,
which is a distinction GitHub has no way to express.

Useful access is untouched, which matters more than the denial. The triage agent still
does its job:

In [ ]:
AGENT_PREFIX=prtriagejava $ASK "$Q" | tail -14

In [ ]:
$PART8/scripts/check-report.sh /tmp/run-code.txt || true

## What that was

1. A developer built and shipped an agent from approved parts, in Java, and it never held
   a credential for the system it talks to.
2. The same job went from a dozen and a half round trips to two on one field, and the
   raw data stopped passing through the model on the way.
3. Two agents share one GitHub integration and have different permissions, enforced on
   their own identities, and the one that may not merge cannot even see the tool.

None of the three happened in the agent's code.

> AgentRegistry supplied the approved integration. kagent deployed and ran the agent.
> agentgateway changed how it used tools and enforced what it could call. The agent
> image stayed the same.

---

## Appendix: the other two settings

Skip this in a short slot. `toolMode` has four values, and they are two independent
choices rather than four flavours:

| | schemas fetched on demand | all schemas up front |
|---|---|---|
| one operation per call | `Search` | `Standard` |
| one program, many operations | `CodeSearch` | `Code` |

- **`Search`** is discovery. 44 tools become `get_tool` and `invoke_tool`: names stay in
  `get_tool`'s description, schemas are fetched on demand. It cuts tool count but not
  round trips, because it still invokes one operation per call.
- **`Code`** is execution, which is what step 5 showed. It cuts round trips and pays
  6,302 tokens to carry all 44 signatures in `run_code`'s description.
- **`CodeSearch`** is both, at 1,300 tokens and two round trips, which is why the demo
  uses it. `invoke_tool` disappears entirely.

Both settings that withhold information from the model failed the first time this ran, in
the same way: the model guessed instead of asking. `Search` sent `per_page` for `perPage`
and folded the owner into `repo` as `"owner/repo"`, and each wrong guess retried the
largest call in the job. `CodeSearch` did the same and then wrote an `else if` chain whose
`Array.isArray(checks.check_runs)` guard swallowed the approval test, reporting eight
unapproved pull requests as ready to merge.

One line in the approved skill fixed both:

> Call `get_tool` for a tool before you first `invoke_tool` it, and use the argument
> names it gives back. Do not guess them.

So the choice is not really which setting is fastest. Pick `Code` when you want the
signatures always present and cannot rely on guidance. Pick `CodeSearch` when a registry
is feeding your agents versioned rules and you want the cheapest context. Pick `Search`
when a task touches one or two tools and a program is overkill.

Run the two step 5 did not:

In [ ]:
for m in Search Code; do
  for ns in agentgateway-system kagent; do
    $KC -n $ns patch enterpriseagentgatewaybackend github-mcp \
      --type=merge -p "{\"spec\":{\"entMcp\":{\"toolMode\":\"$m\"}}}" >/dev/null
  done
  $PART8/scripts/wait-for-mode.sh "$m" >/dev/null
  $PART8/scripts/reload-agent.sh >/dev/null
  echo "### $m"
  AGENT_PREFIX=prtriagejava $ASK "$Q" > /tmp/run-$m.txt 2>&1
  $PART8/scripts/trace-cost.sh /tmp/run-$m.txt
  grep -E "^Ready to merge" /tmp/run-$m.txt | head -1
done

## Reset / teardown

Back to a clean, re-runnable state. The Secret and backend stay, so `setup.sh` is a
one-off.

In [ ]:
$KC -n agentgateway-system delete enterpriseagentgatewaypolicy github-readonly --ignore-not-found
$KC -n kagent delete enterpriseagentgatewaypolicy github-per-agent --ignore-not-found
for ns in agentgateway-system kagent; do
  $KC -n $ns patch enterpriseagentgatewaybackend github-mcp \
    --type=merge -p '{"spec":{"entMcp":{"toolMode":"Standard"}}}' 2>/dev/null || true
done
$PART8/scripts/reload-agent.sh
echo "✓ back to Standard mode, no policy"

Remove Part 8 entirely:

In [ ]:
arctl delete deployment releasejava 2>/dev/null || true
arctl delete agent releasejava 2>/dev/null || true
arctl delete deployment prtriagejava 2>/dev/null || true
arctl delete agent prtriagejava 2>/dev/null || true
arctl delete mcpserver github-mcp 2>/dev/null || true
arctl delete skill release-report 2>/dev/null || true
$KC -n agentgateway-system delete enterpriseagentgatewaypolicy github-readonly --ignore-not-found
$KC -n agentgateway-system delete enterpriseagentgatewaybackend github-mcp --ignore-not-found
$KC -n agentgateway-system delete httproute github-mcp --ignore-not-found
$KC -n agentgateway-system delete secret github-mcp-pat --ignore-not-found
$KC -n kagent delete enterpriseagentgatewaypolicy github-per-agent --ignore-not-found
$KC delete -f $PART8/yaml/15-github-waypoint.yaml --ignore-not-found
$KC -n kagent delete secret github-mcp-pat --ignore-not-found
echo "✓ Part 8 removed"